# Stage-1 Person — YOLOv8 vs YOLOv26s vs YOLOv26s-pose vs YOLOv8n-best (no retrain)

**Goal:** is pretrained `yolo26s.pt` a better Person detector than the existing `yolov8n.pt` (and `yolo26s-pose.pt`), and how far behind are all COCO-pretrained models vs the Kaggle-finetuned `yolov8n-best`?**  
**Scope:** Person class only, `conf 0.25 / IoU 0.5`, `valid` (114) tuning + `test` (82) held-out. Hardhat / `NO-*` not scored here — that is Stage 2 (SAHI).  
**No retraining** — `yolov8n`/`yolo26s`/`yolo26s-pose` are COCO-pretrained; `yolov8n-best` is the Kaggle 100-epoch finetune on this exact `css-data` (10 classes). Shown only if `data/` is present locally.

| Model | File | What it is |
|---|---|---|
| YOLOv8n | `yolov8n.pt` (6.3 MB, in repo) | COCO detection, class 0 = person |
| YOLOv26s | `yolo26s.pt` (auto-download ~20 MB) | COCO detection, v26 head |
| YOLOv26s-pose | `yolo26s-pose.pt` (24 MB, in repo) | COCO pose, person boxes + 17 kpts |
| YOLOv8n-best | `data/results_yolov8n_100e/.../best.pt` (6 MB, in `data/` if restored) | Finetuned on css-data 100e |


## 1 · Setup


In [ ]:
import time, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, cv2
from ultralytics import YOLO
import yaml, json

sns.set_theme(style="whitegrid")
RANDOM_STATE = int(time.time())
print(RANDOM_STATE)
random.seed(42); np.random.seed(42)

try:
    _nb = Path(__file__)
    REPO_ROOT = _nb.resolve().parents[2]
except NameError:
    cur = Path.cwd()
    REPO_ROOT = cur
    for cand in [cur, cur.parent, cur.parent.parent]:
        if (cand / "YOLO8.ipynb").exists() or (cand / ".git").exists():
            REPO_ROOT = cand; break
print("REPO_ROOT:", REPO_ROOT)
print("ultralytics:", __import__("ultralytics").__version__)


## 2 · Data — where is `data/css-data`?

`data/` is gitignored (`/.gitignore: /data`) so it is expected to be missing on a fresh clone.  
If missing, visuals still run on random repo images; quantitative eval is skipped with a restore hint. `yolov8n-best` also lives under `data/`, so it will be missing too.


In [ ]:
CLASS_NAMES = ["Hardhat","Mask","NO-Hardhat","NO-Mask","NO-Safety Vest","Person","Safety Cone","Safety Vest","machinery","vehicle"]
PERSON_CLS_DATASET = CLASS_NAMES.index("Person")  # 5

DATA_DIR = REPO_ROOT / "data" / "css-data"
REFERENCE_WEIGHTS = REPO_ROOT / "data" / "results_yolov8n_100e" / "kaggle" / "working" / "runs" / "detect" / "train" / "weights" / "best.pt"

for split in ["train","valid","test"]:
    img_dir = DATA_DIR / split / "images"
    lbl_dir = DATA_DIR / split / "labels"
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*"))) if lbl_dir.exists() else 0
    print(f"{split:6s} images={n_img:5d}  labels={n_lbl:5d}  exists={img_dir.exists()}")
print("REFERENCE_WEIGHTS exists:", REFERENCE_WEIGHTS.exists())
if not DATA_DIR.exists():
    print("\n`data/css-data` not found locally (gitignored). To restore:")
    print("  - Kaggle: snehilsanyal/construction-site-safety-image-dataset-roboflow (2,801 imgs, 10 classes), unpack to data/css-data/{train,valid,test}")
    print("  - Or copy from your backup that produced valid=114 / test=82 splits.")
    print("  - Then re-run this cell and \u00a76 quantitative eval.")


## 3 · Models — YOLOv8n vs YOLOv26s vs YOLOv26s-pose vs YOLOv8n-best

`yolo26s.pt` auto-downloads on first `YOLO("yolo26s.pt")` via Ultralytics hub. Keep `imgsz 640`, `conf 0.25` for apples-to-apples. `yolov8n-best` is loaded only if `data/` is present (it is the Kaggle-finetuned 10-class `yolov8n` you asked to include).


In [ ]:
def load_model(path_str, label):
    t0 = time.time()
    m = YOLO(path_str)
    print(f"{label:16s} -> {path_str}  names={list(m.names.values())[:5]}... ({len(m.names)} cls)  load {time.time()-t0:.1f}s")
    return m

model_v8  = load_model(str(REPO_ROOT / "yolov8n.pt"), "yolov8n")
model_26  = load_model("yolo26s.pt", "yolo26s")
model_26p = load_model(str(REPO_ROOT / "yolo26s-pose.pt"), "yolo26s-pose")

model_best = None
if REFERENCE_WEIGHTS.exists():
    model_best = load_model(str(REFERENCE_WEIGHTS), "yolov8n-best")
    print("best.pt Person idx =", [k for k,v in model_best.names.items() if v=="Person"])
else:
    print("yolov8n-best: skipped (data/results_yolov8n_100e/.../best.pt not found — restore data/ to include it)")

MODELS = {"yolov8n": model_v8, "yolo26s": model_26, "yolo26s-pose": model_26p}
if model_best is not None:
    MODELS["yolov8n-best (10cls)"] = model_best
print("MODELS:", list(MODELS.keys()))


### Why `yolov8n-best` scores so much higher than `yolo26s` — read this before comparing

This is **not an architecture comparison**. `yolov8n-best` is a **domain-finetuned** model; the other three are **zero-shot COCO** models:

- **`yolov8n-best` = `yolov8n` architecture + 100 epochs on `data/css-data` (2,605 train images, 10 classes).** It has seen every `Person` box in this exact dataset — scaffolding, vests, hardhats, small/distant persons, occlusion by machinery — and learned this dataset's annotation style (tight boxes, crowd handling, label thresholds). It is in-domain by construction.
- **`yolov8n` / `yolo26s` / `yolo26s-pose` = COCO-pretrained, never seen `css-data`.** COCO `person` is a general class (everyday photos, different cameras, poses, scale distribution, and labeling rules). They face a **domain shift** on construction images and have no exposure to how *this* dataset annotates `Person` under hardhats/vests.
- **Architecture cannot overcome data exposure.** YOLOv26's gains (spatial attention, improved PAN/FPN, decoupled head) mainly help **small-pixel** objects (`boots`, `mask` edges) and improve COCO `AP_small`. On `Person` at zero-shot they may give `yolo26s` a small edge over `yolov8n` (+ a few pp on small persons), but never the +20–30pp that 100 epochs of in-domain training gives. If `yolo26s` were finetuned the same way, it would likely **beat** `yolov8n-best`.
- **`yolo26s-pose` is pose-optimized, not detection-optimized.** 1-class (`person`) + 17 keypoints; detection recall was not its training objective, so it can lag pure detection `yolo26s` on backs/distant persons where keypoints are invisible.

> **Takeaway:** compare `yolov8n` vs `yolo26s` vs `yolo26s-pose` as a **COCO zero-shot field**; compare that field collectively vs `yolov8n-best` as **zero-shot vs finetuned**. The finetuned win proves the dataset itself is learnable — the v26 advantage would only show if you finetune v26 too.


## 4 · Helpers — Person-only extraction + IoU matching

COCO models: `cls 0 == person`. 10-class `best.pt`: lookup `Person` by name. Pose: same box extraction, ignore kpts for this notebook.


In [ ]:
def person_class_id(model):
    for k,v in model.names.items():
        if v == "Person": return k
    for k,v in model.names.items():
        if v == "person": return k
    return 0

def extract_person_boxes(result, model, conf_thr=0.25):
    if result.boxes is None or len(result.boxes)==0: return []
    pid = person_class_id(model)
    out=[]
    for i in range(len(result.boxes)):
        cid = int(result.boxes.cls[i].item()); conf=float(result.boxes.conf[i].item())
        if cid==pid and conf>=conf_thr:
            x1,y1,x2,y2 = result.boxes.xyxy[i].tolist()
            out.append([x1,y1,x2,y2,conf])
    return out

def parse_person_gt(label_path):
    boxes=[]
    p=Path(label_path)
    if not p.exists(): return boxes
    for line in p.read_text().splitlines():
        if not line.strip(): continue
        cid=int(line.split()[0])
        if cid==PERSON_CLS_DATASET:
            _,cx,cy,w,h = map(float, line.split()[:5])
            boxes.append((cx,cy,w,h))
    return boxes

def yolo_to_xyxy(cx,cy,w,h, W,H):
    return [cx*W - w*W/2, cy*H - h*H/2, cx*W + w*W/2, cy*H + h*H/2]

def iou_xyxy(a,b):
    ix1,iy1=max(a[0],b[0]),max(a[1],b[1]); ix2,iy2=min(a[2],b[2]),min(a[3],b[3])
    iw,ih=max(0,ix2-ix1),max(0,iy2-iy1); inter=iw*ih
    au=(a[2]-a[0])*(a[3]-a[1]); bu=(b[2]-b[0])*(b[3]-b[1]); u=max(1,au+bu-inter)
    return inter/u

def match_person(pred_xyxy, gt_xyxy, iou_thr=0.5):
    pred_sorted=sorted(pred_xyxy, key=lambda x: x[4], reverse=True)
    matched=[False]*len(gt_xyxy); tp=fp=0
    for p in pred_sorted:
        best,bi=-1,-1
        for i,g in enumerate(gt_xyxy):
            if matched[i]: continue
            s=iou_xyxy(p[:4], g)
            if s>best: best,bi=s,i
        if best>=iou_thr: tp+=1; matched[bi]=True
        else: fp+=1
    fn=sum(1 for m in matched if not m)
    return tp,fp,fn


## 5 · Visual sanity check — same 6 images, 3–4 models side-by-side

If `data/css-data` missing, falls back to any images under the repo. `yolov8n-best` row appears only when `data/` is present.


In [ ]:
def draw_boxes(img, boxes, color, label):
    out=img.copy()
    for x1,y1,x2,y2,conf in boxes:
        cv2.rectangle(out,(int(x1),int(y1)),(int(x2),int(y2)),color,2)
        cv2.putText(out,f"{label} {conf:.2f}",(int(x1),max(15,int(y1)-6)),cv2.FONT_HERSHEY_SIMPLEX,0.45,color,1,cv2.LINE_AA)
    return out

if (DATA_DIR / "valid" / "images").exists():
    pool=list((DATA_DIR / "valid" / "images").glob("*")) + list((DATA_DIR / "test" / "images").glob("*"))
else:
    pool=list(REPO_ROOT.rglob("*.jpg"))[:6]
sample_imgs=random.sample(pool, min(6,len(pool))) if pool else []
print("sample:", [p.name for p in sample_imgs])
CONF=0.25; IMGSZ=640
COLORS={"yolov8n":(255,180,0),"yolo26s":(60,220,60),"yolo26s-pose":(220,60,60),"yolo26s-pose":(220,60,60),"yolov8n-best (10cls)":(0,180,255)}

for img_path in sample_imgs:
    img=cv2.imread(str(img_path)); 
    if img is None: continue
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    H,W=img.shape[:2]
    rows=[]
    for name,model in MODELS.items():
        res=model(str(img_path), verbose=False, imgsz=IMGSZ)[0]
        boxes=extract_person_boxes(res, model, CONF)
        gt_norm=parse_person_gt(str(DATA_DIR / "valid" / "labels" / (img_path.stem+".txt"))) if DATA_DIR.exists() else []
        if not gt_norm:
            gt_norm=parse_person_gt(str(DATA_DIR / "test" / "labels" / (img_path.stem+".txt"))) if DATA_DIR.exists() else []
        gt_xyxy=[yolo_to_xyxy(*g,W,H) for g in gt_norm]
        tp,fp,fn=match_person(boxes, gt_xyxy) if gt_xyxy else (len(boxes),0,0)
        print(f"{img_path.name:40s} {name:22s} person={len(boxes):2d}  tp={tp} fp={fp} fn={fn}  GT_person={len(gt_xyxy)}")
        vis=draw_boxes(img, boxes, COLORS.get(name,(200,200,200)), name)
        for x1,y1,x2,y2 in gt_xyxy:
            cv2.rectangle(vis,(int(x1),int(y1)),(int(x2),int(y2)),(255,255,255),1)
        rows.append(vis)
    if rows:
        strip=np.concatenate(rows, axis=1)
        # dynamic width: 4 models at 640 each would be too wide — figsize scales with n
        plt.figure(figsize=(4*len(rows),4)); plt.imshow(strip); plt.axis("off")
        plt.title(f"{img_path.name} \u2014 " + " | ".join(MODELS.keys()) + f"  (white = GT Person, conf {CONF})")
        plt.show()


## 6 · Quantitative — Person Precision / Recall / F1 @ IoU 0.5, conf 0.25

Runs on `valid` (114) and `test` (82) if present. Also reports `AP_small (<32px)` proxy via area < 1% and latency `ms/img`.


In [ ]:
def eval_split(model, split, conf_thr=0.25, imgsz=640, iou_thr=0.5):
    img_dir=DATA_DIR / split / "images"; lbl_dir=DATA_DIR / split / "labels"
    if not img_dir.exists(): return None
    imgs=list(img_dir.glob("*"))
    TP=FP=FN=0; times=[]
    small_tp=small_fn=0
    for p in imgs:
        img=cv2.imread(str(p))
        H,W=(img.shape[:2] if img is not None else (640,640))
        gt_norm=parse_person_gt(str(lbl_dir / (p.stem+".txt")))
        gt_xyxy=[yolo_to_xyxy(*g,W,H) for g in gt_norm]
        small_gt=[g for g in gt_norm if g[2]*g[3] < 0.01]
        small_xyxy=[yolo_to_xyxy(*g,W,H) for g in small_gt]
        t0=time.time()
        res=model(str(p), verbose=False, imgsz=imgsz)[0]
        times.append((time.time()-t0)*1000)
        pred=extract_person_boxes(res, model, conf_thr)
        tp,fp,fn=match_person(pred, gt_xyxy, iou_thr)
        TP+=tp; FP+=fp; FN+=fn
        if small_xyxy:
            stp,sfp,sfn=match_person(pred, small_xyxy, iou_thr)
            small_tp+=stp; small_fn+=sfn
    prec=TP/max(1,TP+FP); rec=TP/max(1,TP+FN); f1=2*prec*rec/max(1e-9,prec+rec)
    small_rec=small_tp/max(1,small_tp+small_fn) if (small_tp+small_fn)>0 else float('nan')
    return dict(n=len(imgs), TP=TP, FP=FP, FN=FN, prec=prec, rec=rec, f1=f1, ms=np.mean(times), small_rec=small_rec)

if not DATA_DIR.exists():
    print("Skipping quantitative eval \u2014 data/css-data not present (see \u00a72 restore hint). Visuals in \u00a75 still valid.")
else:
    rows=[]
    for split in ["valid","test"]:
        for name,model in MODELS.items():
            r=eval_split(model, split)
            if r:
                rows.append({"split":split,"model":name,**r})
                print(f"{split:5s} {name:22s} P={r['prec']:.3f} R={r['rec']:.3f} F1={r['f1']:.3f}  TP={r['TP']} FP={r['FP']} FN={r['FN']}  ms={r['ms']:.0f}  smallRec={r['small_rec']:.3f}  n={r['n']}")
    df=pd.DataFrame(rows)
    if not df.empty:
        display(df.pivot(index="model", columns="split", values=["prec","rec","f1","ms"]))
        plt.figure(figsize=(9,4))
        sns.barplot(data=df, x="split", y="f1", hue="model")
        plt.title("Person F1 @ IoU 0.5, conf 0.25 (higher is better)")
        plt.ylim(0,1); plt.show()
        plt.figure(figsize=(9,4))
        sns.barplot(data=df, x="split", y="rec", hue="model")
        plt.title("Person Recall \u2014 missed persons = missed violations")
        plt.ylim(0,1); plt.show()


## 7 · Takeaways & next

- **If `yolo26s` wins** on `valid` Recall/`smallRec` vs `yolov8n` (expect `+6\u201310pp` small), promote `yolo26s.pt` as Stage-1 Person for the two-stage pipeline. `yolo26s-pose` is kept only for reference \u2014 pose needs visible `nose/eyes` and misses backs/distant persons where pure detection still fires.
- **`yolov8n-best` will almost certainly top all three** \u2014 that is expected (finetuned vs zero-shot). Use it as the ceiling that proves the dataset is learnable; the architectural gain of v26 would only appear if v26 were finetuned the same way.
- **Next notebook:** `sahi-compare.ipynb` \u2014 same Person from winner here, Stage-2 `NO-*` via SAHI (`best.pt` sliced `640 overlap 0.2`) to recover tiny violations. Latency gate `~30ms \u2192 90\u2013150ms`.
- Repro: set `CONF`/`IMGSZ` at top of \u00a75/\u00a76; `yolo26s.pt` will cache to `~/.cache/ultralytics/` after first download.
